In [1]:
import pandas as pd
import requests
import re

# Path to your dataset
CSV_PATH = "sift-data.xlsx"

In [2]:
def extract_components(name_str):
    try:
        refseq = re.search(r"(NM_\d+(\.\d+)?)", name_str).group(1)
        gene = re.search(r"\((.*?)\)", name_str).group(1)
        coding_change = re.search(r"c\.\-?\d+[ACGT]>[ACGT]", name_str).group(0)
        return refseq, gene, coding_change
    except:
        return None, None, None


In [3]:

def strip_version(nm_id):
    return nm_id.split(".")[0] 


In [4]:

def get_ensembl_transcript(nm_id):
    base_id = strip_version(nm_id)
    url = f"https://rest.ensembl.org/xrefs/name/homo_sapiens/{base_id}?content-type=application/json"
    r = requests.get(url)
    if r.ok:
        for entry in r.json():
            if entry.get("id", "").startswith("ENST"):
                return entry["id"]
    return None



In [5]:

def get_sift_score(transcript_id, coding_change):
    variant = f"{transcript_id}:{coding_change}"
    url = f"https://rest.ensembl.org/variant_recoder/human/{variant}?content-type=application/json"
    r = requests.get(url)
    if not r.ok:
        return None
    data = r.json()
    for entry in data:
        for t in entry.get("transcript_consequences", []):
            if "sift_prediction" in t:
                return {
                    "prediction": t["sift_prediction"],
                    "score": t["sift_score"]
                }
    return None


In [6]:
def get_transcripts_by_gene(gene_symbol):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene_symbol}?expand=1;content-type=application/json"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        return [t['id'] for t in data.get('Transcript', []) if t['id'].startswith("ENST")]
    return []


In [7]:
df = pd.read_excel(CSV_PATH)

# only missense variants
df = df[df["Molecular consequence"].str.contains("missense", case=False, na=False)]

results = []
for _, row in df.iterrows():
    name = row["Name"]
    refseq, gene, coding = extract_components(name)
    if not coding:
        continue

    ensembl_transcript = get_ensembl_transcript(refseq)

    if ensembl_transcript:
        transcripts_to_try = [ensembl_transcript]
    else:
        print(f"⚠️ No match for {refseq}, trying fallback with gene: {gene}")
        transcripts_to_try = get_transcripts_by_gene(gene)

    found = False
    for tx in transcripts_to_try:
        sift = get_sift_score(tx, coding)
        if sift:
            results.append({
                "Gene": gene,
                "RefSeq": refseq,
                "Transcript": tx,
                "c. Change": coding,
                "SIFT Prediction": sift["prediction"],
                "SIFT Score": sift["score"]
            })
            found = True
            break

    if not found:
        print(f"no SIFT result for {gene}:{coding}")

    sift = get_sift_score(ensembl_transcript, coding)
    if sift:
        results.append({
            "Gene": gene,
            "RefSeq": refseq,
            "Transcript": ensembl_transcript,
            "c. Change": coding,
            "SIFT Prediction": sift["prediction"],
            "SIFT Score": sift["score"]
        })
    else:
        print(f"no SIFT result for {ensembl_transcript}:{coding}")

out = pd.DataFrame(results)
out.to_csv("sift_predictions.csv", index=False)
print("saved to sift_predictions.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'sift-data.xlsx'